In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import os
import seaborn as sns
from tqdm import tqdm
import torch

from utils import set_seed

import mintflow

In [ ]:
slide_id = "120" # 210(617k), 221(370k), 222(140k), 231(298k), 242(474k)
adata_load_path = "/data2/a330d/datasets/crc/raw_zenodo"
adata_save_path = "/data2/a330d/datasets/crc/processed"

In [ ]:
set_seed(0)

# Get dataset

In [ ]:
adata = sc.read(
    f"{adata_load_path}/crc_{slide_id}.h5ad"
)

In [ ]:
adata.obs_names_make_unique()

label_to_coarse = {
    "epi1": "Epithelial",
    "epi2": "Epithelial",
    "epi3": "Epithelial",
    "epi4": "Epithelial",
    
    "fib1": "Fibroblast",
    "fib2": "Fibroblast",
    
    "EC": "Endothelial",
    "SMC": "Smooth_muscle",
    
    "BC": "B_cell",
    "PC_IgA": "Plasma_cell",
    "PC_IgG": "Plasma_cell",
    "PC_IgM": "Plasma_cell",
    
    "TC": "T_cell",
    
    "mye1": "Myeloid",
    "mye2": "Myeloid",
    
    "mast": "Mast_cell",
}

adata.obs["coarse_type"] = adata.obs['ist'].map(label_to_coarse)
labels_key = 'coarse_type'
domains_key = 'typ'
batch_key = 'sid'
adata = adata[~adata.obs[domains_key].isna()] # NOTE: Interesting to annotate?
adata = adata[~adata.obs[labels_key].isna()]

sc.pp.filter_cells(adata, min_counts=3)
sc.pp.filter_genes(adata, min_counts=3)

In [ ]:
adata.obs[labels_key] = adata.obs[labels_key].astype('category')
adata.obsm['spatial'] = adata.obs[['CenterX_global_px', 'CenterY_global_px']].values
adata.layers['counts'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, layer='counts', flavor='seurat_v3', n_top_genes=2000, subset=True)

In [ ]:
adata.X = adata.layers['counts'].copy() # NOTE: use raw counts for training

In [ ]:
adata[adata.obs.coarse_type == "Fibroblast"].obs.typ.value_counts()

In [ ]:
adata.obs.typ.value_counts()

In [ ]:
adata.obs["sliceID"] = f"slide_{slide_id}"
adata.obs["batchID"] = f"slide_{slide_id}"

adata.obs["sliceID"] = adata.obs["sliceID"].astype("category")
adata.obs["batchID"] = adata.obs["batchID"].astype("category")

In [ ]:
# For cells in obs[typ].str.contains("CRC"), append "_CRC" to their obs[coarse_type]
adata.obs['mintflow_broad_types'] = adata.obs[labels_key].copy()
adata.obs['mintflow_broad_types'] = adata.obs['mintflow_broad_types'].astype('str')
adata.obs['mintflow_broad_types'] = adata.obs['mintflow_broad_types'].where(
    ~adata.obs['typ'].str.contains("CRC"),
    adata.obs['mintflow_broad_types'] + "_CRC"
)
labels_key = 'mintflow_broad_types'

In [ ]:
x = 0.05  # fraction of cells to keep (e.g., 20%)

n_cells = adata.n_obs
n_subsample = int(n_cells * x)

# Randomly choose cell indices
np.random.seed(42)  # for reproducibility
subsample_idx = np.random.choice(n_cells, n_subsample, replace=False)

# Create the subsampled AnnData
adata = adata[subsample_idx].copy()

In [ ]:
adata.write_h5ad(f"{adata_save_path}/crc_{slide_id}.h5ad")

# Mintflow

In [ ]:
num_epochs = 1 #50
batch_size = 2048

labels_key = 'coarse_type' #'mintflow_broad_types'
patient_id = 'sid'
n_neighbors = 5
x_pos = 'CenterX_global_px'
y_pos = 'CenterY_global_px'
use_wandb = 'False'
n_particles = 10    # Number of predictions per cell for mintflow
path_output_files = f"/data2/a330d/data/ood/trained/crc_{slide_id}/mintflow/"
os.makedirs(path_output_files, exist_ok=True)

## Training

In [ ]:
config_data_train, config_data_evaluation, config_model, config_training = \
    mintflow.get_default_configurations(
        num_tissue_sections_training=1,
        num_tissue_sections_evaluation=1
    )

In [ ]:
train_file = f"{adata_save_path}/crc_{slide_id}.h5ad"

In [ ]:
# configure tissue section 1 =========
config_data_train['list_tissue']['anndata1']['file'] = train_file
#   the absolute path to anndata object of tissue section 1 on disk.

config_data_train['list_tissue']['anndata1']['obskey_cell_type'] = labels_key
#   meaning that for the 1st tissue section, cell type labels are provided in `broad_celltypes` column of `adata.obs`.

config_data_train['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = patient_id
#   meaning that for the 1st tissue section, tissue section ID (i.e. slice ID) is provided in `sid` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['obskey_x'] = x_pos
#   meaning that for the 1st tissue section, spatial x coordinates are provided in `CenterX_global_px` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['obskey_y'] = y_pos
#   meaning that for the 1st tissue section, spatial y coordinates are provided in `CenterY_global_px` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['obskey_biological_batch_key'] = patient_id
#   meaning that for the 1st tissue section, batch identifier is provided in `info_id` column of `adata.obs`

config_data_train['list_tissue']['anndata1']['config_dataloader_train']['width_window'] = batch_size
#   For tissue section one, the crop size of the customized dataloader desribed in Supplementary Fig. 16 of the paper.
#   The larger this number, the larger the tissue crops, and the bigger the subset of cells in each training iteration.
#      This implies that more GPU memory would be required during training.
#   In this notebook after calling `mintflow.setup_data` in Sec 4 the crop(s) are shown on tissue, 
#      with some information on image title which can help you tune this parameter.
#   In the manuscript we used `width_window` values between 300 and 800 depending on dataset.

config_data_train['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
    'n_neighs': n_neighbors,
    'set_diag': 'False',
    'delaunay': 'False',
}
#   The parameters for creating the neighbourhood graph for training tissue section 1

In [ ]:
config_data_evaluation['list_tissue']['anndata1']['file'] = train_file
config_data_evaluation['list_tissue']['anndata1']['obskey_cell_type'] = labels_key
config_data_evaluation['list_tissue']['anndata1']['obskey_sliceid_to_checkUnique'] = patient_id
config_data_evaluation['list_tissue']['anndata1']['obskey_x'] = x_pos
config_data_evaluation['list_tissue']['anndata1']['obskey_y'] = y_pos
config_data_evaluation['list_tissue']['anndata1']['obskey_biological_batch_key'] = patient_id
config_data_evaluation['list_tissue']['anndata1']['config_dataloader_test']['width_window'] = batch_size
config_data_evaluation['list_tissue']['anndata1']['config_neighbourhood_graph'] = {
    'n_neighs': n_neighbors,
    'set_diag': 'False',
    'delaunay': 'False',
}

In [ ]:
config_data_train = mintflow.verify_and_postprocess_config_data_train(config_data_train) 

In [ ]:
config_data_evaluation = mintflow.verify_and_postprocess_config_data_evaluation(config_data_evaluation)

In [ ]:
config_model = mintflow.verify_and_postprocess_config_model(
    config_model,
    num_tissue_sections=1
)

In [ ]:
config_training['num_training_epochs'] = num_epochs

config_training['flag_enable_wandb'] = use_wandb

config_training['flag_finaleval_createanndata_alltissuescombined'] = 'True'

In [ ]:
config_training = mintflow.verify_and_postprocess_config_training(config_training)

In [ ]:
dict_all4_configs = {
    "config_data_train": config_data_train,
    "config_data_evaluation": config_data_evaluation,
    "config_model": config_model,
    "config_training": config_training,
}

data_mintflow = mintflow.setup_data(dict_all4_configs=dict_all4_configs)

In [ ]:
model = mintflow.setup_model(
    dict_all4_configs=dict_all4_configs,
    data_mintflow=data_mintflow
)

In [ ]:
trainer = mintflow.Trainer(
    dict_all4_configs=dict_all4_configs,
    model=model,
    data_mintflow=data_mintflow
)

In [ ]:
def train_mintflow(model, trainer, data_mintflow, dict_all4_configs, path_output_files):
    for epoch in tqdm(range(dict_all4_configs["config_training"]["num_training_epochs"]), desc="Training Epochs"):
        trainer.train_one_epoch()

        mintflow.dump_checkpoint(
            model=model,
            data_mintflow=data_mintflow,
            dict_all4_configs=dict_all4_configs,
            path_dump=os.path.join(path_output_files, "checkpoint_epoch_{}.pt".format(epoch)),
        )

In [ ]:
import torch
import psutil
import os
import time
import threading
import numpy as np
import pandas as pd
from pathlib import Path

class TrainingProfiler:

    def __init__(self, interval=1.0):
        self.interval = interval
        self.process = psutil.Process(os.getpid())

        self.ram_usage = []
        self.vram_usage = []

        self._running = False
        self.thread = None

    def _sample_memory_usage(self):
        while self._running:
            ram = self.process.memory_info().rss / 1024**3
            self.ram_usage.append(ram)

            if torch.cuda.is_available():
                vram = torch.cuda.memory_allocated() / 1024**3
            else:
                vram = 0

            self.vram_usage.append(vram)

            time.sleep(self.interval)

    def start(self):
        self._running = True
        self.start_time = time.time()

        self.thread = threading.Thread(target=self._sample_memory_usage)
        self.thread.daemon = True
        self.thread.start()

    def stop(self):
        self._running = False
        self.thread.join()

        self.end_time = time.time()

    def summary(self):

        return {
            "total_train_time_sec": self.end_time - self.start_time,
            "avg_ram_gb": np.mean(self.ram_usage),
            "avg_vram_gb": np.mean(self.vram_usage),
            "peak_vram_gb": torch.cuda.max_memory_allocated() / 1024**3
        }
    

def profile_training(
    train_fn,
    model_name,
    num_epochs,
    dataset_size,
    adata_path,
    *args,
    csv_path="training_stats.csv",
    **kwargs
):

    profiler = TrainingProfiler(interval=1)

    torch.cuda.reset_peak_memory_stats()

    profiler.start()

    train_fn(*args, **kwargs)

    profiler.stop()

    results = {}
    results["model_name"] = model_name
    results["num_epochs"] = num_epochs
    results["dataset_size"] = dataset_size
    results["adata_path"] = adata_path
    results.update(profiler.summary())

    df = pd.DataFrame([results])

    if Path(csv_path).exists():
        df.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df.to_csv(csv_path, index=False)

    return results


profile_training(
    lambda: train_mintflow(model=model, 
                           trainer=trainer, 
                           data_mintflow=data_mintflow, 
                           dict_all4_configs=dict_all4_configs, 
                           path_output_files=path_output_files),
    model_name="mintflow",
    num_epochs=num_epochs,
    dataset_size=adata.n_obs,
    adata_path=train_file,
    csv_path="../results/training_stats.csv"
)

# Eval via in-silico GEX prediction

In [ ]:
import scanpy as sc

adata_eval = sc.read_h5ad(eval_file)

In [ ]:
result_generation = mintflow.generate_insilico_ST_data(
    adata=adata_eval,
    obskey_celltype='coarse_type',
    obspkey_neighbourhood_graph='spatial_connectivities',
    device=device,
    batch_index_trainingdata=0,
    num_generated_realisations=n_particles,
    model=checkpoint_mintflow['model'],
    data_mintflow=checkpoint_mintflow['data_mintflow'],
    dict_all4_configs=checkpoint_mintflow['dict_all4_configs'],
    estimate_spatial_sizefactors_on_sections=[0]
)

In [ ]:
for k, v in result_generation['list_generated_realisations_ie_expressions'][0].items():
    print(k, v.shape)

In [ ]:
adata_eval.obsm['mintflow_recon_x'] = np.stack(
    [realisation['MintFlow_Generated_Xint'] + realisation['MintFLow_Generated_Xmic'] for realisation in result_generation['list_generated_realisations_ie_expressions']]
).mean(0)

In [ ]:

sns.histplot(adata_eval.obsm['mintflow_recon_x'].sum(axis=1))

In [ ]:
x_hat = adata_eval.obsm['mintflow_recon_x']

In [ ]:
for k, v in result_generation['list_generated_realisations_ie_expressions'][0].items():
    print(k, v.shape)

In [ ]:
np.save("/data2/a330d/datasets/mintflow_xhat_202", x_hat)

# Counterfactual cell types

In [ ]:
batch_size = 90000

In [ ]:
# Your checkpoint with model and data
#trained_checkpoint_path = os.path.join(path_output_files, f"checkpoint_epoch_{num_epochs-1}.pt")
trained_checkpoint_path = os.path.join(path_output_files, f"checkpoint_epoch_{10}.pt")

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

checkpoint_mintflow = torch.load(
    trained_checkpoint_path,
    map_location='cpu',
    weights_only=False
)
checkpoint_mintflow['model'].to(device)
print("Loaded the checkpoint.")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
adata = sc.read(
    f"{adata_save_path}/crc_{slide_id}.h5ad"
)

In [ ]:
kwargs_neighbourhood_graph = {
    'spatial_key': 'spatial',
    'library_key': None,
    'set_diag': False,
    'delaunay': False,
    'n_neighs': 5
}
adata.uns = {}

import squidpy as sq
sq.gr.spatial_neighbors(
    adata=adata,
    **kwargs_neighbourhood_graph
)

In [ ]:
num_generated_realisations = 5
n_cells = adata.n_obs
n_genes = adata.n_vars

recon_x = np.zeros((n_cells, n_genes), dtype=np.float32)

graph = adata.obsp["spatial_connectivities"]

indices = np.arange(adata.n_obs)

for start in tqdm(range(0, len(indices), batch_size), desc="Generating in-silico data in batches"):
    batch_idx = indices[start:start + batch_size]

    # include neighbors
    neighbor_idx = graph[batch_idx].nonzero()[1]
    all_idx = np.unique(np.concatenate([batch_idx, neighbor_idx]))

    adata_batch = adata[all_idx].copy()

    with torch.no_grad():
        res = mintflow.generate_insilico_ST_data(
            adata=adata_batch,
            obskey_celltype=labels_key,
            obspkey_neighbourhood_graph="spatial_connectivities",
            device=device,
            batch_index_trainingdata=0,
            num_generated_realisations=num_generated_realisations,
            model=checkpoint_mintflow["model"],
            data_mintflow=checkpoint_mintflow["data_mintflow"],
            dict_all4_configs=checkpoint_mintflow["dict_all4_configs"],
            estimate_spatial_sizefactors_on_sections=[0]
        )

    # compute mean reconstruction across realizations
    recon_batch = np.stack([
        r["MintFlow_Generated_Xint"] + r["MintFLow_Generated_Xmic"]
        for r in res["list_generated_realisations_ie_expressions"]
    ]).mean(0)

    # map results back to global indices
    pos = np.searchsorted(all_idx, batch_idx)

    recon_x[batch_idx] = recon_batch[pos]

    torch.cuda.empty_cache()

In [ ]:
adata.obsm['recon_x'] = recon_x

In [ ]:
results[0]['list_generated_realisations_ie_expressions'][num_realizations].keys()

## For single cell type (demo)

In [ ]:
target_celltype = "Fibroblast"

mask_target_ct = adata.obs[labels_key] == target_celltype
mask_crc = adata.obs["typ"].str.contains("CRC", na=False)
mask_crc_ct = mask_target_ct & mask_crc
mask_healthy_ct = mask_target_ct & ~mask_crc

crc_idx = np.where(mask_crc_ct)[0]
healthy_idx = np.where(mask_healthy_ct)[0]

sampled_healthy = np.random.choice(healthy_idx, size=len(crc_idx), replace=True)
adata_perturbed = adata.copy()

adata_perturbed.X[crc_idx] = adata.X[sampled_healthy]

In [ ]:
all_categories = sorted(
    set(adata.obs[labels_key].unique()) |
    set(adata_perturbed.obs[labels_key].unique())
)

# Make the column categorical in both adatas
adata.obs[labels_key] = adata.obs[labels_key].astype("category")
adata_perturbed.obs[labels_key] = adata_perturbed.obs[labels_key].astype("category")

adata.obs[labels_key].cat.set_categories(all_categories)
adata_perturbed.obs[labels_key].cat.set_categories(all_categories)

In [ ]:
palette = sc.pl.palettes.default_20[:len(all_categories)]
# or use `sc.pl.palettes.vega_20` if you want different style

palette_dict = dict(zip(all_categories, palette))

In [ ]:
sc.pl.spatial(
    adata,
    spot_size=100,
    color=labels_key,
    palette=palette_dict
)

In [ ]:
sc.pl.spatial(
    adata_perturbed,
    color=labels_key,
    spot_size=100,
    palette=palette_dict
)

In [ ]:
kwargs_neighbourhood_graph = {
    'spatial_key': 'spatial',
    'library_key': None,
    'set_diag': False,
    'delaunay': False,
    'n_neighs': 10
}
adata_perturbed.uns = {}

import squidpy as sq
sq.gr.spatial_neighbors(
    adata=adata_perturbed,
    **kwargs_neighbourhood_graph
)

In [ ]:
num_generated_realisations = 5

result_generation_perturbed = mintflow.generate_insilico_ST_data(
    adata=adata_perturbed,
    obskey_celltype=labels_key,
    obspkey_neighbourhood_graph='spatial_connectivities',
    device=device,
    batch_index_trainingdata=0,
    num_generated_realisations=num_generated_realisations,
    model=checkpoint_mintflow['model'],
    data_mintflow=checkpoint_mintflow['data_mintflow'],
    dict_all4_configs=checkpoint_mintflow['dict_all4_configs'],
    estimate_spatial_sizefactors_on_sections=[0]
)

In [ ]:
for k, v in result_generation_perturbed['list_generated_realisations_ie_expressions'][0].items():
    print(k, v.shape)

In [ ]:
adata.obsm['perturbed_x'] = np.stack(
    [realisation['MintFlow_Generated_Xint'] + realisation['MintFLow_Generated_Xmic'] for realisation in result_generation_perturbed['list_generated_realisations_ie_expressions']]
).mean(0)

### Plots

In [ ]:
adata_ctrl = adata[mask_healthy_ct].copy()
adata_target = adata[mask_crc_ct].copy()
adata_cf = adata[mask_crc_ct].copy()

In [ ]:
# Place relevant model preds in 'recon_x'
adata_cf.obsm['recon_x'] = adata_cf.obsm['perturbed_x']

In [ ]:
adata_ctrl.obs['group'] = 'control'
adata_target.obs['group'] = 'target'
adata_cf.obs['group'] = 'counterfactual'

In [ ]:
adata_merged = ad.concat([adata_ctrl, adata_target, adata_cf], axis=0)

In [ ]:
mintflow_dict = {}

mintflow_dict[target_celltype] = adata_merged

In [ ]:
from counterfactual_analysis import get_de_correlations

In [ ]:
k = 50
plot = True
use_recon = True
normalize_counts = False

In [ ]:
mintflow_lfc, _ = get_de_correlations(mintflow_dict, k=k, plot=plot, use_recon=use_recon, normalize_counts=normalize_counts)

In [ ]:
df_base_path = '../results/mintflow_crc'

In [ ]:
results_df = pd.DataFrame(mintflow_lfc)
summary_df = results_df.groupby("celltype")[["pearson", "spearman"]].mean().reset_index()

# save for later use
summary_df.to_csv(os.path.join(df_base_path, "mintflow_by_celltype_correlations.csv"), index=False)
summary_df

# Predict for all cell types

In [ ]:
target_celltypes = ["Fibroblast", "Endothelial", "Myeloid", "T_cell", "Epithelial"]
mintflow_dict = {}
num_generated_realisations = 5

graph = adata.obsp["spatial_connectivities"]

for target_ct in tqdm(target_celltypes, desc="Target Cell Types"):

    mask_target_ct = adata.obs[labels_key] == target_ct
    mask_crc = adata.obs["typ"].str.contains("CRC", na=False)

    mask_crc_ct = mask_target_ct & mask_crc
    mask_healthy_ct = mask_target_ct & ~mask_crc

    crc_idx = np.where(mask_crc_ct)[0]
    healthy_idx = np.where(mask_healthy_ct)[0]

    n_genes = adata.n_vars
    perturbed_x = np.zeros((adata.n_obs, n_genes), dtype=np.float32)

    for start in tqdm(range(0, len(crc_idx), batch_size), desc="Generating in-silico data in batches"):

        batch_cells = crc_idx[start:start + batch_size]

        # include neighbors
        neighbor_idx = graph[batch_cells].nonzero()[1]
        all_idx = np.unique(np.concatenate([batch_cells, neighbor_idx]))

        adata_batch = adata[all_idx].copy()
        adata_batch.uns = {}

        # map global indices → batch indices
        batch_pos = np.searchsorted(all_idx, batch_cells)

        # choose healthy replacements
        sampled_healthy = np.random.choice(healthy_idx, size=len(batch_cells), replace=True)

        # replace gene expression for CRC cells
        adata_batch.X[batch_pos] = adata.X[sampled_healthy]

        with torch.no_grad():

            res = mintflow.generate_insilico_ST_data(
                adata=adata_batch,
                obskey_celltype=labels_key,
                obspkey_neighbourhood_graph="spatial_connectivities",
                device=device,
                batch_index_trainingdata=0,
                num_generated_realisations=num_generated_realisations,
                model=checkpoint_mintflow["model"],
                data_mintflow=checkpoint_mintflow["data_mintflow"],
                dict_all4_configs=checkpoint_mintflow["dict_all4_configs"],
                estimate_spatial_sizefactors_on_sections=[0]
            )

        recon_batch = np.stack([
            r["MintFlow_Generated_Xint"] + r["MintFLow_Generated_Xmic"]
            for r in res["list_generated_realisations_ie_expressions"]
        ]).mean(0)

        perturbed_x[batch_cells] = recon_batch[batch_pos]

        torch.cuda.empty_cache()

    # store results
    adata_cf = adata[mask_crc_ct].copy()
    adata_cf.obsm["recon_x"] = perturbed_x[mask_crc_ct]

    adata_ctrl = adata[mask_healthy_ct].copy()
    adata_target = adata[mask_crc_ct].copy()

    adata_ctrl.obs["group"] = "control"
    adata_target.obs["group"] = "target"
    adata_cf.obs["group"] = "counterfactual"

    adata_merged = ad.concat([adata_ctrl, adata_target, adata_cf])

    mintflow_dict[target_ct] = adata_merged

In [ ]:
from counterfactual_analysis import get_de_correlations

In [ ]:
k = 50
plot = True
use_recon = False
normalize_counts = True

In [ ]:
mintflow_lfc, _ = get_de_correlations(mintflow_dict, k=k, plot=plot, use_recon=use_recon, normalize_counts=normalize_counts)

In [ ]:
df_base_path = '../results'

In [ ]:
results_df = pd.DataFrame(mintflow_lfc)
# Rename celltype to holdout_celltype
results_df = results_df.rename(columns={"celltype": "holdout_celltype", "prec@50": "precision"})
model_name = 'mintflow'
model_name += '-recon' if use_recon else ''
model_name += '-normalized' if normalize_counts else ''
results_df['model_name'] = model_name
results_df['sid'] = slide_id
results_df['edistance_cells'] = 0
results_df['edistance_latents'] = 0

In [ ]:
results_df

In [ ]:
# save for later use
df_save_path = f"{model_name}_{slide_id}_by_celltype_correlations.csv"
results_df.to_csv(os.path.join(df_base_path, df_save_path), index=False)